Linear Probing for AdaptViT
=================================
Trains a classification head on top of pruned ViT models generated by the
pruning pipeline, and evaluates top-1 accuracy on a held-out test set.
Also runs the same linear probing on the unpruned base models for comparison.

The transformer backbone is frozen throughout — only the classification head
is trained (20 epochs, AdamW).

Pipeline:
  1. Clone SnapViT and set up the Pets dataset
  2. Define the fixed train/val/test split (reproduced via fixed seeds)
  3. Linear probe each pruned model variant across all sparsity levels
  4. Linear probe each unpruned base model

Author: Vishnu PS

## Setup

In [ ]:
import os
import re
import random
import sys
import shutil
import subprocess
from tqdm import tqdm
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

!git clone https://github.com/WalterSimoncini/SnapViT.git
%cd SnapViT
!pip install -r requirements.txt


## Prepare Dataset — Oxford-IIIT Pets (37 classes)

Downloaded directly from the Oxford servers and organised into per-breed
subfolders. A fixed random seed (2026) is used to ensure the test split is
identical across all runs, so accuracy numbers are comparable across models.

In [ ]:
import os
import re

LOCAL_IMAGES = "/content/local_pets"
RAW_DIR      = "/tmp/pets_raw"
TAR_PATH     = "/tmp/pets.tar.gz"

os.makedirs(LOCAL_IMAGES, exist_ok=True)

existing_breeds = [d for d in os.listdir(LOCAL_IMAGES)
                   if os.path.isdir(os.path.join(LOCAL_IMAGES, d))]

if len(existing_breeds) >= 37:
    print(f"Already organised for this session ({len(existing_breeds)} breeds) — skipping")
else:
    print("Downloading Oxford-IIIT Pets images (~800 MB)...")
    !wget -q --show-progress -O {TAR_PATH} \
        https://thor.robots.ox.ac.uk/~vgg/data/pets/images.tar.gz

    print("Extracting...")
    os.makedirs(RAW_DIR, exist_ok=True)
    !tar -xf {TAR_PATH} -C {RAW_DIR} --strip-components=1

    print("Organising into breed subfolders...")
    pattern = re.compile(r'^(.+)_\d+\.jpg$', re.IGNORECASE)
    for fname in os.listdir(RAW_DIR):
        m = pattern.match(fname)
        if not m:
            continue
        breed = m.group(1)
        breed_dir = os.path.join(LOCAL_IMAGES, breed)
        os.makedirs(breed_dir, exist_ok=True)
        src = os.path.join(RAW_DIR, fname)
        dst = os.path.join(breed_dir, fname)
        if not os.path.exists(dst):
            os.rename(src, dst)

    breeds       = [d for d in os.listdir(LOCAL_IMAGES)
                    if os.path.isdir(os.path.join(LOCAL_IMAGES, d))]
    total_images = sum(len(os.listdir(os.path.join(LOCAL_IMAGES, b))) for b in breeds)
    print(f"\nDATASET READY!")
    print(f"Breeds   : {len(breeds)}")
    print(f"Images   : {total_images}")
    print(f"Location : {LOCAL_IMAGES}")

## Linear probing — pruned models
For each model and sparsity level, loads the pruned backbone from Drive,
resets the classifier to NUM_CLASSES outputs, and fine-tunes only the head.
If head_mlp.pt already exists for a config, fine-tuning is skipped and only
test accuracy is recomputed — useful for resuming interrupted runs.

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split
import random
import sys
import os
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

IMAGES_PATH = LOCAL_IMAGES
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


dataset = datasets.ImageFolder(IMAGES_PATH, transform=transform)


# split 80/20 into train and val using a separate seed.
indices = list(range(len(dataset)))
random.seed(2026)
random.shuffle(indices)

test_size = int(0.2 * len(dataset))
trainval_indices = indices[:-test_size]
test_indices     = indices[-test_size:]

trainval_dataset = Subset(dataset, trainval_indices)
test_dataset     = Subset(dataset, test_indices)

train_size = int(0.8 * len(trainval_dataset))
val_size   = len(trainval_dataset) - train_size

train_set, val_set = random_split(
    trainval_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)
val_loader   = DataLoader(val_set,   batch_size=128, shuffle=False,
                          num_workers=8, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_dataset, batch_size=128, shuffle=False,
                          num_workers=8, pin_memory=True, persistent_workers=True)

print(f"Total images     : {len(dataset)}")
print(f"Train            : {len(train_set)}")
print(f"Validation       : {len(val_set)}")
print(f"Test (held-out)  : {len(test_dataset)}")
print(f"Classes          : {len(dataset.classes)}")

CLASS_NAMES = dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

# Validation + Test Accuracy Helper
def eval_accuracy(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total * 100

# Configuration
BASE_PATH      = f'/content/drive/MyDrive/PhD_Works/SnapViT_Q2'
MODELS_ROOT    = f'{BASE_PATH}/model_outputs'

PRUNING_LEVELS = [
    (0.15, 0.0), (0.25, 0.1), (0.35, 0.2),
    (0.45, 0.3), (0.55, 0.4), (0.65, 0.5),
]

MODEL_NAMES = sorted([
    d for d in os.listdir(MODELS_ROOT)
    if os.path.isdir(os.path.join(MODELS_ROOT, d))
])
print(f"Models found: {MODEL_NAMES}\n")

print(f"Running for selected models: {MODEL_NAMES}\n")

# Main Loop
sys.path.insert(0, '/content/SnapViT')

for MODEL_NAME in MODEL_NAMES:
    MODEL_FOLDER = os.path.join(MODELS_ROOT, MODEL_NAME)
    print(f"\n{'#'*75}")
    print(f"PROCESSING MODEL: {MODEL_NAME}")
    print(f"{'#'*75}")

    for MLP_PRUNING_LEVEL, HEAD_PRUNING_LEVEL in PRUNING_LEVELS:
        PRUNING_FOLDER    = f"mlp-{MLP_PRUNING_LEVEL}-heads-{HEAD_PRUNING_LEVEL}"
        pruned_model_path = os.path.join(MODEL_FOLDER, PRUNING_FOLDER, "full_pruned_model.pt")
        HEAD_PATH_NEW     = os.path.join(MODEL_FOLDER, PRUNING_FOLDER, "head_mlp.pt")

        if not os.path.exists(pruned_model_path):
            print(f"  [{PRUNING_FOLDER}] Pruned model not found — skipping")
            continue

        if os.path.exists(HEAD_PATH_NEW):
            print(f"  [{PRUNING_FOLDER}] head_mlp.pt already exists — skipping fine-tuning")
        else:
            print(f"\n  → Fine-tuning: {PRUNING_FOLDER}")

            pruned_model = torch.load(pruned_model_path, map_location=device, weights_only=False)
            pruned_model.reset_classifier(NUM_CLASSES)
            pruned_model.to(device)

            for p in pruned_model.blocks.parameters():
                p.requires_grad = False
            for p in pruned_model.head.parameters():
                p.requires_grad = True

            optimizer = torch.optim.AdamW(pruned_model.head.parameters(), lr=1e-3, weight_decay=0.01)
            criterion = torch.nn.CrossEntropyLoss()

            print(f"  Trainable params: {sum(p.numel() for p in pruned_model.head.parameters()):,}")

            for epoch in range(20):
                pruned_model.train()
                total_loss = 0.0
                pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/20", leave=False)

                for imgs, labels in pbar:
                    imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                    optimizer.zero_grad()
                    loss = criterion(pruned_model(imgs), labels)
                    loss.backward()
                    optimizer.step()
                    total_loss += loss.item()
                    pbar.set_postfix({"loss": f"{loss.item():.4f}"})

                avg_loss = total_loss / len(train_loader)
                val_acc = eval_accuracy(pruned_model, val_loader)
                print(f"  Epoch {epoch+1:2d}/20 — Loss: {avg_loss:.4f} | Val Acc: {val_acc:.2f}%")

            pruned_model.eval()
            torch.save(pruned_model.head.state_dict(), HEAD_PATH_NEW)
            print(f"  → New head saved as: head_mlp_new.pt")

        print(f"  → Evaluating on held-out test set... ", end="")
        pruned_model = torch.load(pruned_model_path, map_location=device, weights_only=False)
        pruned_model.reset_classifier(NUM_CLASSES)
        pruned_model.head.load_state_dict(torch.load(HEAD_PATH_NEW, map_location=device))
        pruned_model = pruned_model.to(device)
        pruned_model.eval()

        test_acc = eval_accuracy(pruned_model, test_loader)

        print(f"Test Accuracy: {test_acc:.2f}%")

        del pruned_model
        torch.cuda.empty_cache()

print("\n" + "="*85)
print("ALL PROCESSING COMPLETED!")
print("Fine-tuning was skipped when head_mlp_new.pt already existed.")
print("Test accuracy was computed for every pruning level.")
print("="*85)

## Linear probing — unpruned base models
Same training setup as above but loads the full pretrained model directly
from timm instead of a pruned checkpoint. Provides the accuracy baseline
against which pruned variants are compared.
TIMM_MODEL_NAMES maps our internal folder names to the correct timm model
identifiers — update this dict if adding new model variants.

In [ ]:
TIMM_MODEL_NAMES = {
    'vit_tiny_r':    'timm/vit_tiny_r_s16_p8_224.augreg_in21k_ft_in1k',
    'augreg_vits16': 'timm/vit_small_patch16_224.augreg_in21k_ft_in1k',
    'augreg_vitb16': 'timm/vit_base_patch16_224.augreg_in21k_ft_in1k',
    'dino_vits16':   'timm/vit_small_patch16_224.dino',
    'dino_vitb16':   'timm/vit_base_patch16_224.dino',
    'deit3_vits16':  'timm/deit3_small_patch16_224.fb_in22k_ft_in1k',
    'deit3_vitb16':  'timm/deit3_base_patch16_224.fb_in22k_ft_in1k',
    'siglip_vitb16':  'timm/ViT-B-16-SigLIP2',
}

for MODEL_NAME in MODEL_NAMES:
    if MODEL_NAME not in TIMM_MODEL_NAMES:
        print(f"  Skipping {MODEL_NAME} — no TIMM mapping found")
        continue

    timm_model_id = TIMM_MODEL_NAMES[MODEL_NAME]
    MODEL_FOLDER = os.path.join(MODELS_ROOT, MODEL_NAME)
    BASE_HEAD_PATH = os.path.join(MODEL_FOLDER, "head_mlp_base.pt")

    print(f"\n{'#'*80}")
    print(f"PROCESSING BASE MODEL: {MODEL_NAME}")
    print(f"   TIMM ID: {timm_model_id}")
    print(f"{'#'*80}")

    if os.path.exists(BASE_HEAD_PATH):
        print(f"  → head_mlp_base.pt already exists — skipping fine-tuning")
    else:
        print(f"  → Fine-tuning classifier head on base model...")

        import timm
        base_model = timm.create_model(timm_model_id, pretrained=True, num_classes=NUM_CLASSES)
        base_model = base_model.to(device)

        for name, param in base_model.named_parameters():
            if not (name.startswith('head.') or name.startswith('classifier.')):
                param.requires_grad = False

        trainable_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
        print(f"  Trainable params (head only): {trainable_params:,}")

        optimizer = torch.optim.AdamW(
            [p for p in base_model.parameters() if p.requires_grad],
            lr=1e-3,
            weight_decay=0.01
        )
        criterion = torch.nn.CrossEntropyLoss()

        for epoch in range(20):
            base_model.train()
            total_loss = 0.0
            pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/20", leave=False)

            for imgs, labels in pbar:
                imgs = imgs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                optimizer.zero_grad()
                outputs = base_model(imgs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                total_loss += loss.item()
                pbar.set_postfix({"loss": f"{loss.item():.4f}"})

            avg_loss = total_loss / len(train_loader)
            val_acc = eval_accuracy(base_model, val_loader)
            print(f"  Epoch {epoch+1:2d}/20 — Loss: {avg_loss:.4f} | Val Acc: {val_acc:.2f}%")

        if hasattr(base_model, 'head'):
            torch.save(base_model.head.state_dict(), BASE_HEAD_PATH)
        elif hasattr(base_model, 'classifier'):
            torch.save(base_model.classifier.state_dict(), BASE_HEAD_PATH)
        else:
            torch.save(base_model.state_dict(), BASE_HEAD_PATH)
            print("  Saved full model state dict (head attribute not found)")

        print(f"  → Base head saved as: head_mlp_base.pt")

    print(f"  → Evaluating on held-out test set... ", end="")

    import timm
    base_model = timm.create_model(timm_model_id, pretrained=True, num_classes=NUM_CLASSES)
    base_model = base_model.to(device)

    head_state = torch.load(BASE_HEAD_PATH, map_location=device)

    if hasattr(base_model, 'head'):
        base_model.head.load_state_dict(head_state)
    elif hasattr(base_model, 'classifier'):
        base_model.classifier.load_state_dict(head_state)

    base_model.eval()
    test_acc = eval_accuracy(base_model, test_loader)

    print(f"Test Accuracy: {test_acc:.2f}%")

    del base_model
    torch.cuda.empty_cache()

print("\n" + "="*90)
print("BASE MODEL LINEAR PROBING COMPLETED SUCCESSFULLY!")
print("• head_mlp_base.pt saved in each model folder")
print("="*90)